# Relativistic kinematics and jet clustering

FeynKit's Python kinematics use double precision, caller-defined energy units, component order \((E,p_x,p_y,p_z)\), and the mostly-minus metric \((+,-,-,-)\).

In [ ]:
from math import cos, cosh, sin, sinh
import symbolica.community.feynkit as fk

## Three- and four-momenta

Construct an on-shell four-vector from a spatial momentum and a mass. The invariant mass check below makes the metric convention explicit.

In [ ]:
spatial = fk.ThreeMomentum(3.0, 4.0, 0.0)
p = spatial.on_shell(12.0)
{
    "components": p.components(),
    "pT": p.pt,
    "mass_squared": p.mass_squared,
    "mass": p.mass,
    "rapidity": p.rapidity,
}

In [ ]:
q = fk.FourMomentum(5.0, 1.0, 2.0, 3.0)
expected = p.energy * q.energy - p.px * q.px - p.py * q.py - p.pz * q.pz
{"dot_product": p.dot(q), "component_formula": expected}

## Lorentz transformations

Boosts require a dimensionless three-velocity with norm below one. Rotations and boosts provide explicit inverse operations, which makes invariant checks straightforward.

In [ ]:
rest = fk.FourMomentum(5.0, 0.0, 0.0, 0.0)
boost = fk.Boost(fk.ThreeMomentum(0.6, 0.0, 0.0))
boosted = boost.apply(rest)
recovered = boost.apply_inverse(boosted)
{
    "rest": rest.components(),
    "boosted": boosted.components(),
    "recovered": recovered.components(),
    "invariant_mass_squared": boosted.mass_squared,
}

In [ ]:
unit_x = fk.ThreeMomentum(1.0, 0.0, 0.0)
unit_y = fk.Rotation.quarter_turn(fk.Axis.Z).apply_three(unit_x)
unit_y

## Angular distances

Azimuth is wrapped consistently, and `delta_r` uses rapidity for four-momenta (pseudorapidity for three-momenta).

In [ ]:
a = fk.FourMomentum(20.0, 10.0, 0.0, 10.0)
b = fk.FourMomentum(20.0, 0.0, 10.0, -10.0)
{
    "delta_phi": a.delta_phi(b),
    "delta_R": a.delta_r(b),
}

## Generalized-\(k_T\) jets

The sequential-recombination algorithms use E-scheme four-vector addition. Results are sorted by decreasing transverse momentum, and constituent indices refer to positions in the input list.

In [ ]:
def massless(
    pt: float, rapidity: float, phi: float
) -> fk.FourMomentum:
    return fk.FourMomentum(
        pt * cosh(rapidity),
        pt * cos(phi),
        pt * sin(phi),
        pt * sinh(rapidity),
    )

particles = [
    massless(80.0, 0.20, 0.10),
    massless(25.0, 0.25, 0.18),
    massless(35.0, -1.10, 2.40),
]

In [ ]:
jet_definition = fk.JetDefinition.anti_kt(radius=0.6, minimum_pt=5.0)
clustered = jet_definition.cluster(particles)
[
    {
        "constituents": jet.constituent_indices,
        "pT": jet.pt,
        "rapidity": jet.rapidity,
        "phi": jet.phi,
    }
    for jet in clustered.jets
]

In [ ]:
definitions = {
    "kt": fk.JetDefinition.kt(0.6, minimum_pt=5.0),
    "cambridge_aachen": fk.JetDefinition.cambridge_aachen(0.6, minimum_pt=5.0),
    "anti_kt": fk.JetDefinition.anti_kt(0.6, minimum_pt=5.0),
}
{
    algorithm: [jet.constituent_indices for jet in definition.cluster(particles).jets]
    for algorithm, definition in definitions.items()
}

## Practical checklist

- Keep one unit system throughout an event.
- Check `mass_squared` after constructing or transforming momenta.
- Choose the jet radius and minimum \(p_T\) in the context of the analysis.
- Preserve the input ordering when interpreting `constituent_indices`.
- Catch `KinematicsError` around user-provided configurations or non-finite event data.